In [ ]:
%pip install openai-agents pydantic mermaid-py nest-asyncio -q

In [2]:
from nest_asyncio import apply
apply()

In [3]:
POWERFUL_MODEL = 'o4-mini'
MODEL = 'gpt-5-mini'

# Blog Post Generator Agent: Planner, Executor, and Critic Loop

## The Brief

In this notebook, you will build a **self-correcting content generation pipeline** — a multi-agent system that plans, writes, and critiques blog posts in an iterative loop until the output reaches a quality threshold.

**What you will build:**
- A **Planner** agent that produces a structured article outline from a topic
- An **Executor/Writer** agent that turns the outline into a full blog post
- A **Critic** agent that scores the article and provides actionable feedback
- An **orchestration loop** in Python that coordinates these three agents, passing feedback from the Critic back to the Writer until the article is good enough (or a maximum number of iterations is reached)

**What you will practice:**
- Creating multiple `Agent` objects with different models and `output_type`
- Calling agents with `Runner.run_sync()` and working with structured Pydantic output
- Writing **imperative orchestration** — a Python loop that controls agent execution, rather than using SDK handoffs or `agent.as_tool()`
- Implementing convergence detection and bounded iteration

**Prerequisites:** You should have completed the Introduction to OpenAI Agents SDK notebook (Lecture 80), where you learned `Agent`, `Runner.run_sync`, `output_type`, `agent.as_tool()`, and handoffs.

## Recommended Architecture

We recommend using **different models for different roles** based on what each step demands:

| Role | Model | Why This Model? |
|------|-------|------------------|
| **Planner** | `o4-mini` (reasoning) | Planning requires structured thinking — organizing sections, anticipating audience needs, ensuring coverage. Reasoning models excel here. |
| **Executor** | `gpt-5-mini` (cheap & fast) | The hard thinking was already done by the Planner. The Writer just needs to flesh out a clear brief. A cheaper model keeps costs low. |
| **Critic** | `o4-mini` (reasoning) | Evaluating quality requires comparing against multiple criteria simultaneously and formulating specific feedback. Another reasoning-heavy task. |

### The Flow

1. The **Planner** receives a topic and generates a detailed article outline (runs once).
2. The **Executor** writes the full article based on that plan.
3. The **Critic** evaluates the article, scores it (1-10), and provides actionable feedback.
4. If the score is below `QUALITY_THRESHOLD` and iterations remain, the Executor revises using the feedback.
5. The loop repeats until convergence or `MAX_ITERATIONS` is reached.

In [4]:
import mermaid as md

md.Mermaid("""
flowchart LR
    A[Topic] --> B[Planner]
    B --> C[Plan]
    C --> D[Executor]
    D --> E[Draft]
    E --> F[Critic]
    F --> G[Feedback]
    G -- If not converged --> D
""")

## Key Design Decisions

### Bounded Iteration
Every iterative agent loop **must** have a hard limit (`MAX_ITERATIONS`). Without it, a perfectionist critic could request revisions forever. We also define a `QUALITY_THRESHOLD` — when the critic scores the article above this threshold, we stop early. Together, these give you **convergence detection with a safety net**.

### Cost Optimization
Not every step needs the most powerful model. Reasoning models cost more but are worth it for planning and critique. Bulk writing is better served by cheaper, faster models. This keeps costs low while maintaining quality where it matters.

### Why Imperative Orchestration (Not Handoffs or Agents-as-Tools)?

In the previous notebook you learned two ways the SDK connects agents: **`agent.as_tool()`** and **handoffs**. You might be wondering: why not use those here?

We *could* build this with a `Coordinator` agent that calls the Planner, Writer, and Critic as tools. But for a **fixed, predictable pipeline** with convergence criteria, **a Python loop is the better choice**:

| | Imperative (Python loop) | SDK-native (handoffs / as_tool) |
|---|---|---|
| **Flow control** | Deterministic — Plan → Write → Critique → Revise, every time | LLM decides — may skip steps or call things out of order |
| **Convergence** | Check score in Python, break when done | Must encode iteration logic in a prompt |
| **Cost control** | You know exactly how many calls happen | LLM may make extra calls |
| **Best for** | Fixed pipelines, iterative refinement | Dynamic routing, open-ended tasks |

**Rule of thumb:** Use the SDK's `Agent` + `Runner.run_sync` to simplify individual agent calls. Keep orchestration in Python when the workflow is structured. Use handoffs/as_tool when the LLM needs to decide the flow (like triage routing).

## Setup

The imports, model configuration, and `ArticleCritique` Pydantic model are provided for you. Run these cells before starting the exercises.

In [5]:
import getpass
import os
from pydantic import BaseModel, Field
from agents import Agent, Runner

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# Model configuration - use reasoning models for planning/critique, cheap model for writing
PLANNER_MODEL = POWERFUL_MODEL  # Reasoning model for high-quality planning
WRITER_MODEL = MODEL            # Cost-effective model for content generation
CRITIC_MODEL = POWERFUL_MODEL   # Reasoning model for quality evaluation
MAX_ITERATIONS = 3              # Maximum revision cycles
QUALITY_THRESHOLD = 8           # Score out of 10 to stop iterating


class ArticleCritique(BaseModel):
    """Structured critique from the reviewer."""
    score: int = Field(description="Quality score from 1 to 10")
    strengths: list[str] = Field(description="List of article strengths")
    weaknesses: list[str] = Field(description="List of article weaknesses")
    specific_feedback: str = Field(description="Detailed, actionable feedback for improvement")
    ready_to_publish: bool = Field(description="Whether the article is publication-ready")


---

## Exercise 1: Define the Three Agents

Create three `Agent` objects — one for each role. You already know how to do this from Lecture 80.

- **`planner_agent`** — Uses `PLANNER_MODEL`. Instructions should tell it to produce a detailed blog post plan (title, intro hook, 3-5 sections with key points, code examples to include, conclusion). Returns raw text (no `output_type`).
- **`writer_agent`** — Uses `WRITER_MODEL`. Instructions should tell it to write a complete, publication-ready blog post in markdown, 800-1200 words. Returns raw text (no `output_type`).
- **`critic_agent`** — Uses `CRITIC_MODEL`. Instructions should tell it to evaluate the article on: plan adherence, technical accuracy, writing quality, practical value, and completeness. Uses `output_type=ArticleCritique` so the output is a parsed Pydantic instance.

In [6]:
# YOUR CODE HERE: Define three agents
#
# planner_agent = Agent(
#     name="Article Planner",
#     instructions="""...""",
#     model=PLANNER_MODEL,
# )
#
# writer_agent = Agent(
#     name="Article Writer",
#     instructions="""...""",
#     model=WRITER_MODEL,
# )
#
# critic_agent = Agent(
#     name="Article Critic",
#     instructions="""...""",
#     model=CRITIC_MODEL,
#     output_type=ArticleCritique,
# )

pass

## Exercise 2: Write the Helper Functions

Write three functions that each wrap a single `Runner.run_sync()` call. The pattern is always: build a prompt string → call `Runner.run_sync(agent, prompt)` → return `result.final_output`.

### `plan_article(topic, audience)`
- Prompt the planner with the topic and target audience.
- Return `result.final_output` (raw text).

### `write_article(plan, feedback=None)`
- Prompt the writer with the plan.
- If `feedback` is provided (from a previous critique), append it to the prompt so the writer addresses it.
- Return `result.final_output` (raw text).

### `critique_article(article, plan, iteration)`
- Prompt the critic with both the original plan and the current article.
- Return `result.final_output` — already an `ArticleCritique` Pydantic instance thanks to `output_type`.

In [7]:
def plan_article(topic, audience="general technical audience"):
    """Use a reasoning model to generate a detailed article plan."""
    # YOUR CODE HERE
    # 1. Build a prompt asking the planner to create a blog post plan for the topic and audience
    # 2. result = Runner.run_sync(planner_agent, prompt)
    # 3. Return result.final_output
    pass


def write_article(plan, feedback=None):
    """Write the full article from a plan, optionally incorporating critic feedback."""
    # YOUR CODE HERE
    # 1. Build a prompt with the plan
    # 2. If feedback is not None, append it to the prompt
    # 3. result = Runner.run_sync(writer_agent, prompt)
    # 4. Return result.final_output
    pass


def critique_article(article, plan, iteration):
    """Evaluate the article and return structured feedback as an ArticleCritique."""
    # YOUR CODE HERE
    # 1. Build a prompt with the original plan and the article (include the iteration number)
    # 2. result = Runner.run_sync(critic_agent, prompt)
    # 3. Return result.final_output (already an ArticleCritique instance!)
    pass

## Exercise 3: The Orchestration Loop

Write `generate_blog_post(topic, audience)` — the function that ties everything together:

1. **Plan once**: Call `plan_article(topic, audience)`.
2. **Write → Critique → Revise loop** (up to `MAX_ITERATIONS`):
   - Call `write_article(plan, feedback)` to produce or revise the article.
   - Call `critique_article(article, plan, iteration)` to evaluate it.
   - If `critique.score >= QUALITY_THRESHOLD` → break (convergence).
   - Otherwise → set `feedback = critique.specific_feedback` for the next iteration.
3. **Return** a dict with: `topic`, `plan`, `article`, `final_score`, `iterations`, `final_critique`.

In [8]:
def generate_blog_post(topic, audience="general technical audience"):
    """
    Full Planner -> Executor -> Critic orchestration loop.

    The loop continues until either:
    1. The critic scores the article >= QUALITY_THRESHOLD (convergence)
    2. MAX_ITERATIONS is reached (safety net)
    """
    # YOUR CODE HERE
    #
    # 1. plan = plan_article(topic, audience)
    #
    # 2. article = None, feedback = None, final_critique = None
    #
    # 3. for iteration in range(1, MAX_ITERATIONS + 1):
    #        article = write_article(plan, feedback=feedback)
    #        critique = critique_article(article, plan, iteration)
    #        final_critique = critique
    #
    #        if critique.score >= QUALITY_THRESHOLD:
    #            print(f"CONVERGED! Score {critique.score}/10 meets threshold of {QUALITY_THRESHOLD}/10")
    #            break
    #        else:
    #            feedback = critique.specific_feedback
    #
    # 4. return {
    #        "topic": topic, "plan": plan, "article": article,
    #        "final_score": final_critique.score,
    #        "iterations": iteration, "final_critique": final_critique
    #    }
    pass

## Test Your Implementation

Run the cells below to test your exercises. If everything is implemented correctly, the pipeline will plan, write, critique, and iterate. If anything returns `None`, go back and implement the exercise functions above.

In [9]:
result = generate_blog_post(
    topic="Building Reliable AI Agents: Lessons from Production Systems",
    audience="software engineers building AI-powered applications"
)

if result:
    print(f"\nFinal Score: {result['final_score']}/10")
    print(f"Iterations: {result['iterations']}/{MAX_ITERATIONS}")
else:
    print("generate_blog_post() returned None. Implement the exercises above and try again!")

generate_blog_post() returned None. Implement the exercises above and try again!


In [10]:
from IPython.display import Markdown, display

if result:
    print(f"\nFinal Score: {result['final_score']}/10")
    print(f"Iterations: {result['iterations']}/{MAX_ITERATIONS}")
    print(f"\n{'='*60}")
    print("FINAL ARTICLE")
    print(f"{'='*60}\n")
    display(Markdown(result['article']))

## Understanding the Cost Tradeoffs

| Role | Model | Cost Level | Justification |
|------|-------|------------|---------------|
| Planner | `o4-mini` | Higher | Planning requires structured reasoning to produce good outlines |
| Writer | `gpt-5-mini` | Lower | Content generation from a clear plan does not need deep reasoning |
| Critic | `o4-mini` | Higher | Quality evaluation requires comparing against multiple criteria simultaneously |

- **The loop typically converges in 2 to 3 iterations.** The first draft is often close, and one round of feedback usually addresses the major gaps.
- **The Planner only runs once** — fixed cost regardless of iterations.
- **Each iteration costs one Writer call + one Critic call** — variable cost is proportional to iterations.

### Tuning for Your Use Case

- **Prioritize quality:** `QUALITY_THRESHOLD = 9`, `MAX_ITERATIONS = 5`
- **Prioritize speed/cost:** `QUALITY_THRESHOLD = 7`, `MAX_ITERATIONS = 2`
- **Quick drafts:** `MAX_ITERATIONS = 1` (plan + write + single critique, no revision)

## Experimenting with Different Settings

Let's try a completely different topic to see how the pipeline adapts. This demonstrates that the system generalizes across different types of content. Try changing the topic or audience to something you are interested in!

In [11]:
# Try with a different topic
result_fast = generate_blog_post(
    topic="5 Python Tips Every Developer Should Know",
    audience="beginner Python programmers"
)

In [12]:
# Display the second article
if result_fast:
    print(f"\nFinal Score: {result_fast['final_score']}/10")
    print(f"Iterations: {result_fast['iterations']}/{MAX_ITERATIONS}")
    print(f"\n{'='*60}")
    print("FINAL ARTICLE")
    print(f"{'='*60}\n")
    display(Markdown(result_fast['article']))

---

## Solutions

Reference implementations for Exercises 1-3. Compare with your approach — there is no single correct answer, but these cover the key points.

### Solution: Exercise 1 — Define the Three Agents

In [18]:
planner_agent = Agent(
    name="Article Planner",
    instructions="""You are a senior content strategist. Create a detailed blog post plan.

Your plan should include:
1. A compelling title
2. An introduction hook
3. 3-5 main sections with key points for each
4. Code examples or practical demonstrations to include
5. A conclusion with key takeaways

Return the plan as clear, structured text. This plan will be given to a writer to produce the full article.""",
    model=PLANNER_MODEL,
)

writer_agent = Agent(
    name="Article Writer",
    instructions="""You are a skilled technical writer. Write a complete, publication-ready blog post.

Requirements:
- Write in a clear, engaging style suitable for a technical blog
- Include code examples where the plan specifies them
- Use markdown formatting (headers, code blocks, bullet points)
- Aim for 800-1200 words
- Make it practical and actionable""",
    model=WRITER_MODEL,
)

critic_agent = Agent(
    name="Article Critic",
    instructions="""I want you to be extremely critical. You are a senior technical editor reviewing a blog post.

Evaluate the article on these criteria:
1. Adherence to the plan (does it cover all planned sections?)
2. Technical accuracy
3. Writing quality (clarity, engagement, flow)
4. Practical value (actionable takeaways, useful code examples)
5. Completeness (introduction, body, conclusion)

Be rigorous but fair. Only give a score of 8+ if the article is genuinely publication-ready.""",
    model=CRITIC_MODEL,
    output_type=ArticleCritique,
)

print(f"Planner: {planner_agent.name} ({planner_agent.model})")
print(f"Writer:  {writer_agent.name} ({writer_agent.model})")
print(f"Critic:  {critic_agent.name} ({critic_agent.model}) -> output_type={critic_agent.output_type.__name__}")

Planner: Article Planner (o4-mini)
Writer:  Article Writer (gpt-5-mini)
Critic:  Article Critic (o4-mini) -> output_type=ArticleCritique


### Solution: Exercise 2 — Helper Functions

In [19]:
def plan_article(topic, audience="general technical audience"):
    """Use a reasoning model to generate a detailed article plan."""
    result = Runner.run_sync(
        planner_agent,
        f"""Create a detailed blog post plan for the following topic.

Topic: {topic}
Target Audience: {audience}"""
    )

    print("=" * 60)
    print("ARTICLE PLAN")
    print("=" * 60)
    print(result.final_output)
    return result.final_output


def write_article(plan, feedback=None):
    """Write the full article from a plan, optionally incorporating critic feedback."""
    prompt = f"""Write a complete, publication-ready blog post based on this plan:

{plan}"""

    if feedback:
        prompt += f"""

IMPORTANT: The following feedback was provided by a reviewer. Address ALL points:

{feedback}"""

    result = Runner.run_sync(writer_agent, prompt)

    print(f"\nArticle generated ({len(result.final_output.split())} words)")
    return result.final_output


def critique_article(article, plan, iteration):
    """Evaluate the article and return structured feedback as an ArticleCritique."""
    result = Runner.run_sync(
        critic_agent,
        f"""ORIGINAL PLAN:
{plan}

ARTICLE (Iteration {iteration}):
{article}"""
    )

    critique = result.final_output

    print(f"\n{'='*60}")
    print(f"CRITIQUE (Iteration {iteration})")
    print(f"{'='*60}")
    print(f"Score: {critique.score}/10")
    print(f"Ready to publish: {critique.ready_to_publish}")
    print(f"Strengths:")
    for s in critique.strengths:
        print(f"  + {s}")
    print(f"Weaknesses:")
    for w in critique.weaknesses:
        print(f"  - {w}")
    print(f"\nFeedback: {critique.specific_feedback}")

    return critique

### Solution: Exercise 3 — The Orchestration Loop

In [20]:
def generate_blog_post(topic, audience="general technical audience"):
    """Full Planner -> Executor -> Critic orchestration loop."""
    print(f"Starting blog post generation for: '{topic}'")
    print(f"Max iterations: {MAX_ITERATIONS}, Quality threshold: {QUALITY_THRESHOLD}/10")
    print(f"Models: Planner={PLANNER_MODEL}, Writer={WRITER_MODEL}, Critic={CRITIC_MODEL}")
    print("=" * 60)

    # Step 1: Plan (runs once)
    print("\n[PHASE 1: PLANNING]")
    plan = plan_article(topic, audience)

    # Step 2: Write -> Critique -> Revise loop
    article = None
    feedback = None
    final_critique = None

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n[PHASE 2: WRITING - Iteration {iteration}/{MAX_ITERATIONS}]")
        article = write_article(plan, feedback=feedback)

        print(f"\n[PHASE 3: CRITIQUE - Iteration {iteration}/{MAX_ITERATIONS}]")
        critique = critique_article(article, plan, iteration)
        final_critique = critique

        # Check convergence
        if critique.score >= QUALITY_THRESHOLD:
            print(f"\n>>> CONVERGED! Score {critique.score}/10 meets threshold of {QUALITY_THRESHOLD}/10. Stopping early.")
            break
        elif iteration < MAX_ITERATIONS:
            print(f"\n>>> Score {critique.score}/10 is below threshold of {QUALITY_THRESHOLD}/10. Revising...")
            feedback = critique.specific_feedback
        else:
            print(f"\n>>> Max iterations reached. Final score: {critique.score}/10 (threshold was {QUALITY_THRESHOLD}/10)")

    return {
        "topic": topic,
        "plan": plan,
        "article": article,
        "final_score": final_critique.score,
        "iterations": iteration,
        "final_critique": final_critique
    }

### Run the Full Pipeline with Solutions

In [21]:
result = generate_blog_post(
    topic="Building Reliable AI Agents: Lessons from Production Systems",
    audience="software engineers building AI-powered applications"
)

print(f"\nFinal Score: {result['final_score']}/10")
print(f"Iterations: {result['iterations']}/{MAX_ITERATIONS}")

Starting blog post generation for: 'Building Reliable AI Agents: Lessons from Production Systems'
Max iterations: 3, Quality threshold: 8/10
Models: Planner=o4-mini, Writer=gpt-5-mini, Critic=o4-mini

[PHASE 1: PLANNING]
ARTICLE PLAN
1. Title  
“Building Rock-Solid AI Agents: Production-Proven Strategies for Reliability”

2. Introduction Hook  
“Imagine your AI assistant confidently scheduling meetings—until it suddenly starts double–booking clients in production. As AI systems become mission-critical, reliability isn’t a luxury: it’s a requirement. In this post, we’ll unpack battle-tested lessons from high-scale production systems to ensure your AI agents run smoothly, predictably, and safely in the wild.”

3. Main Sections  

Section 1: Architecting for Resilience  
 • Microservice-based agent components: separation of prompt preprocessing, model inference, postprocessing, and orchestration  
 • Idempotent operations: design tasks so retries don’t cause duplicate side effects  
 • Ci

In [17]:
print(f"{'='*60}")
print("FINAL ARTICLE")
print(f"{'='*60}\n")
display(Markdown(result['article']))

FINAL ARTICLE



# Engineering Reliable AI Agents: Production-Proven Strategies for Software Engineers

A few years ago a major retailer rolled out an AI chat assistant to help customers with ordering and returns. On Black Friday the assistant silently started returning malformed answers and timeouts. The result: thousands of lost sales, dozens of angry support tickets, and a week of scrambles to pin down a cascade that began in a preprocessing service. Building an agent was the easy part — keeping it reliable under real-world stress proved far harder.

If you’re responsible for production AI agents, you need more than a good model: you need resilient architecture, observability, rigorous testing, and automated delivery practices. This post borrows battle-tested approaches from large-scale software systems and shows practical ways to design, monitor, test, and operate AI agents that stay up and behave predictably.

## Defining Reliability for AI Agents

Reliability for agents spans multiple dimensions. Explicitly define what you mean for your system:

- Uptime & Availability
  - Set SLAs for inference endpoints (e.g., 99.9% monthly uptime).
  - Redundancy: replicate services across zones/regions and use load balancing/auto-scaling.
- Correctness & Robustness
  - Handle malformed or adversarial inputs gracefully — never return system stack traces or leak PII.
  - Use input validation, sanitization, and fallback responses.
- Performance & Latency
  - Define SLOs for tail latency (p95/p99) and throughput. Real-time features may require <300ms p95.
  - Distinguish between latency-sensitive interactive agents and batch/async workloads.
- Safety & Compliance
  - Enforce guardrails for toxic language, hallucinations, or regulatory requirements (PII masking, retention policies).
  - Maintain audit trails of decisions and prompts where required.
- Key Metrics to Track
  - Error rates (4xx/5xx), mean time between failures (MTBF), mean time to recovery (MTTR), p95/p99 latency, throughput, model confidence scores, and safety violations.

## Architecting for Resilience

Design your stack so failures are contained and recoverable.

- Modular Design
  - Split preprocessing, inference, and postprocessing into independent services. Replace or scale them independently.
- Service Isolation
  - Containerize with Docker and deploy via Kubernetes or serverless functions so a single bad component doesn’t take down the whole system.
- State Management
  - Prefer stateless agents where possible. Externalize session or user state into Redis, DynamoDB, or a managed session store.
- Circuit Breakers & Bulkheads
  - Use circuit breakers to fail fast when an upstream model endpoint is unhealthy.
  - Bulkheads isolate resources (CPU, threads, connection pools) so overloaded components don’t exhaust global capacity.

High-level request flow (ASCII diagram):

request → router/orchestrator → [preprocessor] → [inference / model cluster] → [postprocessor] → response  
                       ↘ skill A (timeout)  
                       ↘ skill B (fallback)

Practical tips:
- Limit concurrent in-flight requests per model to protect memory and GPUs.
- Implement timeouts and fallback answers for downstream failures.

## Observability & Real-Time Monitoring

Instrumentation is non-negotiable.

- Logs vs Metrics vs Traces
  - Logs: rich context for failures and inputs (sanitized).
  - Metrics: numeric SLOs (latency, error rate, queue length).
  - Traces: distributed timing for request flows across components.
- Instrumenting Your Agent
  - Use OpenTelemetry to capture spans and export to an OTLP collector.
  - Example (Python + OpenTelemetry):
    ```python
    from opentelemetry import trace
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
    from opentelemetry.sdk.trace.export import BatchSpanProcessor

    trace.set_tracer_provider(TracerProvider())
    tracer = trace.get_tracer(__name__)
    exporter = OTLPSpanExporter(endpoint="otel-collector:4317")
    trace.get_tracer_provider().add_span_processor(BatchSpanProcessor(exporter))

    def handle_request(request):
        with tracer.start_as_current_span("agent-inference"):
            # preprocess
            response = model.predict(request)
            # postprocess
            return response
    ```
- Alerting & Anomaly Detection
  - Set threshold alerts for error rate increases and latency spikes; pair with anomaly detection on rolling baselines.
  - Integrate PagerDuty/Slack for on-call routing and runbooks.
- Dashboards
  - Use Prometheus + Grafana or CloudWatch for SLO dashboards (latency histograms, error rates, model-specific metrics).

## Testing, Validation & Chaos Engineering

Reliability emerges from testing at every level.

- Unit Tests
  - Verify pre/post-processing deterministically. Catch regressions early.
- Integration Tests
  - Mock external APIs (LLM endpoints) and assert shape and safety of responses.
- Simulation & Stress Tests
  - Use k6 or Locust for spike and sustained concurrency tests against your inference endpoint.
- Canary Deployments & A/B Testing
  - Route a small percentage of traffic to new model versions. Monitor safety and latency before gradual rollout.
- Chaos Experiments
  - Simulate network latency, upstream downtime, or resource exhaustion (Chaos Mesh, Gremlin).
- Example Pytest integration test:
  ```python
  import pytest
  from agent import handle_request

  @pytest.mark.parametrize("prompt, expected_prefix", [
    ("Hello, world", "Hi there"),
    ("Summarize:", "Summary:")
  ])
  def test_agent_responses(prompt, expected_prefix):
      response = handle_request({"text": prompt})
      assert response.startswith(expected_prefix)
  ```

Testing tips:
- Run safety checks (toxicity detectors) in CI for candidate model outputs.
- Maintain a synthetic test corpus that reflects real-world customer prompts and edge cases.

## Continuous Delivery, Rollbacks & Governance

Automate deployments and enforce guardrails.

- CI/CD Pipelines
  - Automate builds, tests, security scans, and deploys. Example GitHub Actions snippet:
    ```yaml
    name: Deploy AI Agent
    on: [push]
    jobs:
      build-and-deploy:
        runs-on: ubuntu-latest
        steps:
          - uses: actions/checkout@v3
          - name: Run tests
            run: pytest
          - name: Build Docker image
            run: docker build -t my-ai-agent:${{ github.sha }} .
          - name: Deploy to Kubernetes
            uses: azure/k8s-deploy@v1
            with:
              manifests: deploy/agent-deployment.yaml
              images: myregistry/my-ai-agent:${{ github.sha }}
    ```
- Versioning Models & Code
  - Use semantic versions and store model artifacts + metadata in a model registry (MLflow, S3 with metadata).
- Automated Rollbacks
  - Gate deployments on health checks. Trigger rollbacks automatically when error rate or latency crosses thresholds.
- Access Control & Audit Logging
  - Enforce RBAC for model promotions and keep immutable audit logs of changes to models, prompts, and infra.

## Conclusion — Key Takeaways

- Reliability is multi-dimensional: design for availability, correctness, performance, and safety.
- Observability isn’t optional: instrument at every layer so you can detect and diagnose anomalies quickly.
- Test at scale and in production-like conditions: integration, load, and chaos tests reveal hidden weaknesses.
- Automate your delivery pipeline with clear rollback gates and governance checks.
- Iterate continuously: monitor model drift, collect feedback, and refine both models and infrastructure.

Production reliability comes from engineering discipline more than from any single tool. Treat your agent like a distributed system: partition responsibilities, monitor every interaction, run experiments that break things, and automate recovery. Do that, and your agents will be far more likely to serve customers reliably when it matters most.

## Bonus Exercise: Build a Code Review Agent

Same Planner, Executor, and Critic pattern, but applied to code review instead of blog post writing. This demonstrates that the pattern is domain-agnostic.

### Exercise 4: Build a Code Review Agent with Planner, Executor, and Critic

Adapt the Planner → Executor → Critic loop to **automated code review**:

1. **Plan** the review — analyze the code snippet, decide what to look for (bugs, security, style, performance).
2. **Execute** the review — write a detailed, structured assessment.
3. **Critique** the review itself — score its thoroughness and accuracy, feed improvement suggestions back.

The loop continues until quality exceeds `QUALITY_THRESHOLD` or `MAX_ITERATIONS` is reached. Same orchestration, different domain.

In [ ]:
import mermaid as md

md.Mermaid("""
flowchart LR
    A[Code Snippet] --> B[Planner]
    B --> C[Review Plan]
    C --> D[Executor]
    D --> E[Review]
    E --> F[Critic]
    F --> G[Feedback]
    G -- If not converged --> D
""")

### Pydantic Models

The `CodeReview` model structures the executor's output. The `ReviewCritique` model structures the critic's evaluation. Both are used as `output_type` on their respective agents — no manual JSON schema generation needed.

In [ ]:
class CodeReview(BaseModel):
    """Structured output from the code review executor."""
    score: int = Field(description="Quality score from 1 to 10")
    bugs_found: list[str] = Field(description="List of bugs or issues found")
    improvements: list[str] = Field(description="Suggested improvements")
    security_concerns: list[str] = Field(description="Any security issues")
    ready_for_merge: bool = Field(description="Whether the code is ready to merge")


class ReviewCritique(BaseModel):
    """Structured critique of the review itself."""
    score: int = Field(description="Review quality score from 1 to 10")
    strengths: list[str] = Field(description="What the review got right")
    weaknesses: list[str] = Field(description="What the review missed or got wrong")
    specific_feedback: str = Field(description="Actionable feedback to improve the review")

### Step 1: Create the Three Agents

Define three agents for the code review pipeline:
- A **review planner** (reasoning model) that analyzes the code and produces a review checklist
- A **review executor** (cheap model, `output_type=CodeReview`) that writes the structured review
- A **review critic** (reasoning model, `output_type=ReviewCritique`) that evaluates the review quality

In [ ]:
# YOUR CODE HERE
# Hint: Create three Agent objects:
#
# review_planner = Agent(
#     name="Review Planner",
#     instructions="Analyze the code and produce a prioritized checklist of things to review...",
#     model=PLANNER_MODEL,
# )
#
# review_executor = Agent(
#     name="Code Reviewer",
#     instructions="Write a structured code review based on the plan...",
#     model=WRITER_MODEL,
#     output_type=CodeReview,
# )
#
# review_critic = Agent(
#     name="Review Critic",
#     instructions="Evaluate the quality of the code review...",
#     model=CRITIC_MODEL,
#     output_type=ReviewCritique,
# )
pass

### Step 2: Write the Helper Functions

Write three functions (`plan_review`, `execute_review`, `critique_review`) that call `Runner.run_sync()` on the respective agents. Each function should build the appropriate prompt and return the result.

In [ ]:
def plan_review(code_snippet):
    """Use a reasoning model to create a detailed review plan for the given code."""
    # YOUR CODE HERE
    # result = Runner.run_sync(review_planner, f"Analyze this code...\n{code_snippet}")
    # return result.final_output
    pass


def execute_review(code_snippet, plan, feedback=None):
    """Write a structured code review based on the plan, optionally incorporating feedback."""
    # YOUR CODE HERE
    # Build a prompt with the code, plan, and optional feedback.
    # result = Runner.run_sync(review_executor, prompt)
    # result.final_output is already a CodeReview instance!
    # return result.final_output
    pass


def critique_review(review, code_snippet, plan, iteration):
    """Evaluate the quality of the code review and provide structured feedback."""
    # YOUR CODE HERE
    # result = Runner.run_sync(review_critic, prompt_with_code_plan_review)
    # result.final_output is already a ReviewCritique instance!
    # return result.final_output
    pass

### Step 3: `review_code(code_snippet)` (The Orchestration Loop)

This is the main function that ties everything together, exactly like `generate_blog_post()` from the blog post generator. It should:

1. Call `plan_review()` once.
2. Enter a loop up to `MAX_ITERATIONS`:
   a. Call `execute_review()` (passing feedback if available).
   b. Call `critique_review()` to evaluate the review.
   c. If the critique score meets `QUALITY_THRESHOLD`, break early (convergence).
   d. Otherwise, pass the critique feedback to the next iteration.
3. Return a dictionary with the final review, score, and iteration count.

In [ ]:
def review_code(code_snippet):
    """Full orchestration loop: Plan -> Review -> Critique -> Revise."""
    # YOUR CODE HERE
    # Follow the exact same structure as generate_blog_post().
    #
    # 1. plan = plan_review(code_snippet)
    # 2. Loop from 1 to MAX_ITERATIONS:
    #      review = execute_review(code_snippet, plan, feedback)
    #      critique = critique_review(review, code_snippet, plan, iteration)
    #      if critique.score >= QUALITY_THRESHOLD: break
    #      else: feedback = critique.specific_feedback
    # 3. Return {"review": review, "final_score": critique.score, "iterations": iteration}
    pass

### Test Your Code Review Agent

The sample code below has several intentional problems: SQL injection, plaintext password storage, missing error handling, and no connection cleanup. A thorough review should catch all of them.

In [ ]:
sample_code = '''
def process_user_data(data):
    conn = sqlite3.connect("users.db")
    query = f"SELECT * FROM users WHERE name = '{data[\'name\']}'"
    result = conn.execute(query).fetchall()
    password = data['password']
    with open('passwords.txt', 'a') as f:
        f.write(password)
    return result
'''

review_result = review_code(sample_code)

if review_result:
    print(f"\nFinal Score: {review_result['final_score']}/10")
    print(f"Iterations: {review_result['iterations']}/{MAX_ITERATIONS}")
    print(f"\nBugs found: {review_result['review'].bugs_found}")
    print(f"Security concerns: {review_result['review'].security_concerns}")
    print(f"Improvements: {review_result['review'].improvements}")
    print(f"Ready for merge: {review_result['review'].ready_for_merge}")
else:
    print("review_code() returned None. Implement the functions above and try again!")

## Summary

In this notebook you built a **Planner, Executor, and Critic** multi-agent loop for blog post generation. Key takeaways:

- **Imperative orchestration vs SDK-native.** We wrote the convergence loop in Python rather than using handoffs or `agent.as_tool()`. This gives deterministic flow, easy convergence detection, and precise cost control. Use SDK-native patterns when routing is dynamic; use imperative loops when the workflow is structured.

- **The SDK simplifies agent calls, not workflow logic.** `Runner.run_sync()` replaces `client.responses.create()` + manual parsing. The loop, feedback passing, and convergence remain your code.

- **`output_type` eliminates JSON boilerplate.** The Critic returns a parsed `ArticleCritique` instance directly.

- **Different models for different roles.** Reasoning models (`o4-mini`) for planning/critique, cheaper models (`gpt-5-mini`) for bulk writing.

- **Always bound your loops.** `MAX_ITERATIONS` prevents runaway costs. `QUALITY_THRESHOLD` enables early stopping.

- **The pattern generalizes.** Blog posts, code reviews, report writing, email drafting — any task where iterative refinement improves output quality.